In [1]:
import pandas as pd
import re

# =====================================================
# Match unique case IDs between Metasheet_Analysis.xlsx
# and AUR_ClinicalData_20260427.NO.DATES.bloodID.withBCRv2.xlsx
# =====================================================

metasheet_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.xlsx"

clinical_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/AUR_ClinicalData_20260427.NO.DATES.bloodID.withBCRv2.xlsx"


# -----------------------------
# Helper function: extract case ID
# Example:
# AUP-ANU1-NB1 -> AUP-ANU1
# AUP-AQ0Z     -> AUP-AQ0Z
# -----------------------------
def extract_case_id(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    m = re.search(r"(AUP-[A-Za-z0-9]+)", x)

    return m.group(1) if m else pd.NA


# -----------------------------
# Read Metasheet_Analysis.xlsx
# -----------------------------
meta = pd.read_excel(metasheet_file)

meta["Case_ID"] = meta["Collaborator_sample_id"].apply(extract_case_id)

meta_cases = set(meta["Case_ID"].dropna().unique())


# -----------------------------
# Read AUR clinical data file
# -----------------------------
clinical = pd.read_excel(clinical_file)

# In case duplicate Patient_BCR_Barcode columns exist,
# pandas may rename them as Patient_BCR_Barcode.1, etc.
barcode_cols = [
    c for c in clinical.columns
    if str(c).startswith("Patient_BCR_Barcode")
]

clinical_case_id_df = clinical[barcode_cols].apply(
    lambda col: col.apply(extract_case_id)
)

clinical["Case_ID"] = clinical_case_id_df.bfill(axis=1).iloc[:, 0]

clinical_cases = set(clinical["Case_ID"].dropna().unique())


# -----------------------------
# Match unique cases
# -----------------------------
matched_cases = meta_cases.intersection(clinical_cases)

meta_not_in_clinical = meta_cases - clinical_cases
clinical_not_in_meta = clinical_cases - meta_cases


# -----------------------------
# Print summary
# -----------------------------
print("================ CASE MATCH SUMMARY ================")
print(f"Unique cases in Metasheet_Analysis.xlsx: {len(meta_cases)}")
print(f"Unique cases in AUR clinical data file: {len(clinical_cases)}")
print(f"Number of matched unique cases: {len(matched_cases)}")
print(f"Cases in Metasheet but not in AUR clinical data file: {len(meta_not_in_clinical)}")
print(f"Cases in AUR clinical data file but not in Metasheet: {len(clinical_not_in_meta)}")

================ CASE MATCH SUMMARY ================
Unique cases in Metasheet_Analysis.xlsx: 270
Unique cases in AUR clinical data file: 171
Number of matched unique cases: 164
Cases in Metasheet but not in AUR clinical data file: 106
Cases in AUR clinical data file but not in Metasheet: 7


In [2]:
# =====================================================
# Receptor status availability among matched cases
# =====================================================

print("\n================ RECEPTOR STATUS IN MATCHED CASES ================")

# Keep only rows from matched cases
clinical_matched = clinical[clinical["Case_ID"].isin(matched_cases)].copy()

# Clean receptor status values
def clean_status(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x == "" or x.lower() in ["nan", "na", "n/a", "none"]:
        return pd.NA
    return x

status_cols = ["ER_Status", "PR_Status", "HER2_Status"]

for col in status_cols:
    clinical_matched[col] = clinical_matched[col].apply(clean_status)

# One receptor-status value per unique case
# If a case has multiple rows, take the first non-missing value
def first_non_missing(series):
    vals = series.dropna().unique()
    return vals[0] if len(vals) > 0 else pd.NA

case_receptor_status = (
    clinical_matched
    .groupby("Case_ID", as_index=False)[status_cols]
    .agg(first_non_missing)
)

# Count availability
n_matched_cases = len(matched_cases)

n_er = case_receptor_status["ER_Status"].notna().sum()
n_pr = case_receptor_status["PR_Status"].notna().sum()
n_her2 = case_receptor_status["HER2_Status"].notna().sum()

n_all_3 = (
    case_receptor_status["ER_Status"].notna()
    & case_receptor_status["PR_Status"].notna()
    & case_receptor_status["HER2_Status"].notna()
).sum()

print(f"Total matched unique cases: {n_matched_cases}")
print(f"Matched cases with ER_Status available: {n_er}")
print(f"Matched cases with PR_Status available: {n_pr}")
print(f"Matched cases with HER2_Status available: {n_her2}")
print(f"Matched cases with all 3 receptor statuses available: {n_all_3}")

print("\nER_Status counts among matched cases:")
print(case_receptor_status["ER_Status"].fillna("Missing/blank").value_counts())

print("\nPR_Status counts among matched cases:")
print(case_receptor_status["PR_Status"].fillna("Missing/blank").value_counts())

print("\nHER2_Status counts among matched cases:")
print(case_receptor_status["HER2_Status"].fillna("Missing/blank").value_counts())


================ RECEPTOR STATUS IN MATCHED CASES ================
Total matched unique cases: 164
Matched cases with ER_Status available: 164
Matched cases with PR_Status available: 164
Matched cases with HER2_Status available: 164
Matched cases with all 3 receptor statuses available: 164

ER_Status counts among matched cases:
Positive    136
Negative     26
Unknown       2
Name: ER_Status, dtype: int64

PR_Status counts among matched cases:
Positive    116
Negative     45
Unknown       3
Name: PR_Status, dtype: int64

HER2_Status counts among matched cases:
Negative    135
Positive     23
Unknown       6
Name: HER2_Status, dtype: int64


In [3]:
# =====================================================
# Histological Type availability among matched cases
# =====================================================

print("\n================ HISTOLOGICAL TYPE IN MATCHED CASES ================")

# Keep only rows from matched cases
clinical_matched = clinical[clinical["Case_ID"].isin(matched_cases)].copy()

# Clean Histologic.type values
def clean_histology(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x == "" or x.lower() in ["nan", "na", "n/a", "none"]:
        return pd.NA
    return x

clinical_matched["Histologic.type"] = clinical_matched["Histologic.type"].apply(clean_histology)

# One histology value per unique case
# If a case has multiple rows, take the first non-missing value
def first_non_missing(series):
    vals = series.dropna().unique()
    return vals[0] if len(vals) > 0 else pd.NA

case_histology = (
    clinical_matched
    .groupby("Case_ID", as_index=False)["Histologic.type"]
    .agg(first_non_missing)
)

# Make sure all matched cases are represented
all_matched_cases_df = pd.DataFrame({"Case_ID": sorted(matched_cases)})

case_histology_full = all_matched_cases_df.merge(
    case_histology,
    on="Case_ID",
    how="left"
)

# Count availability
n_matched_cases = len(case_histology_full)
n_hist_available = case_histology_full["Histologic.type"].notna().sum()
n_hist_missing = case_histology_full["Histologic.type"].isna().sum()

print(f"Total matched unique cases: {n_matched_cases}")
print(f"Matched cases with Histologic.type available: {n_hist_available}")
print(f"Matched cases with Histologic.type missing: {n_hist_missing}")

print("\nUnique Histologic.type values among matched cases:")
print(case_histology_full["Histologic.type"].dropna().nunique())

print("\nHistologic.type counts among matched cases:")
print(
    case_histology_full["Histologic.type"]
    .fillna("Missing/blank")
    .value_counts()
)


================ HISTOLOGICAL TYPE IN MATCHED CASES ================
Total matched unique cases: 164
Matched cases with Histologic.type available: 159
Matched cases with Histologic.type missing: 5

Unique Histologic.type values among matched cases:
3

Histologic.type counts among matched cases:
Ductal                      133
Lobular                      17
Mixed Ductal and Lobular      9
Missing/blank                 5
Name: Histologic.type, dtype: int64
